# ASF-UAV-Warning — демонстраційна Monte Carlo модель агентного мультисенсорного виявлення БпЛА
## Demonstrative Monte Carlo model of agentic multisensor UAV detection and public warning

Пакет відтворюваності до статті / Reproducibility package for the article:

> O. Korchenko, D. Prokopovych-Tkachenko, A. Desiatko, I. Azarov, O. Galushchenko, M. Mormul.
> **Agentic Multisensor System for Early Unmanned Aircraft Detection and Public Warning.**
> *Artificial Intelligence* (ISSN 2710-1673), 2026.

---

### ▶ Самодостатня Colab-версія

Це версія **для Google Colab**: код моделі вбудовано безпосередньо в комірки —
жодних зовнішніх файлів чи `!git clone` не потрібно. Достатньо *Runtime → Run all*.
Використовуються лише `numpy`, `pandas`, `matplotlib`, `scikit-learn` (усі є в Colab).

Модульна версія тих самих обчислень (пакет `src/asf_simulation.py`, `src/metrics.py`,
`src/make_figures.py`) разом із `pytest`-тестами відтворюваності доступна в репозиторії
у каталозі `asf-uav-warning/`. Логіка й порядок звертань до генератора випадкових чисел
тут **ідентичні** модульній версії, тому за `SEED = 20260` метрики збігаються біт-у-біт.

**Важливо / Important.** Усі числові результати є **демонстраційними**: вони характеризують
**синтетичну** генеративну модель, параметри якої задані авторами, а не реальні вимірювання.

## 0. Модель, метрики та рисунки (вбудований код)

Три комірки нижче містять той самий код, що й модулі `src/` у репозиторії:
**генеративна модель**, **метрики з bootstrap-CI** та **побудова рисунків**. У Colab
достатньо виконати їх один раз — далі ноутбук лише викликає ці функції.

In [ ]:
# ============================================================
# Модуль 1/3: генеративна модель (еквівалент src/asf_simulation.py)
# ============================================================
"""Генеративна Monte Carlo модель агентного мультисенсорного виявлення БпЛА.

Generative Monte Carlo model of agentic multisensor UAV detection.

Цей модуль містить *лише* синтетичну генеративну модель: параметри сенсорів,
генерацію подій та побудову оцінок трьох архітектур ухвалення рішення. Він не
залежить від ``matplotlib`` чи ``sklearn`` — тільки ``numpy`` та ``pandas``.

Усі числові результати є **демонстраційними**: вони характеризують синтетичну
модель, параметри якої задані авторами статті, а не реальні вимірювання.

Порядок звертань до спільного генератора ``rng`` збережено точно таким, як в
оригінальному ноутбуці, тому за ``SEED = 20260`` метрики відтворюються біт-у-біт.
"""


from dataclasses import dataclass

import numpy as np
import pandas as pd

# ============================================================
# Гіперпараметри відтворюваності / Reproducibility hyperparameters
# ============================================================
SEED = 20260  # фіксоване зерно відтворюваності
N_EVENTS = 64_000  # загальна кількість подій Monte Carlo
N_TEST = 19_200  # розмір тестової частини (30 %)
N_BOOT = 800  # кількість bootstrap-перевибірок для 95 % CI

MODALITIES = ["radar", "rf", "acoustic", "optical"]

# --- Параметри генеративної моделі (відкалібровані під цільові метрики статті) ---
BASE = {"radar": 0.885, "rf": 0.93, "acoustic": 0.84, "optical": 0.90}  # β_m
DSLOPE = {"radar": 0.052, "rf": 0.048, "acoustic": 0.075, "optical": 0.058}  # α_m, 1/км
NOISE = {"radar": 0.185, "rf": 0.13, "acoustic": 0.175, "optical": 0.15}  # σ_m
P_DROP = {"radar": 0.04, "rf": 0.06, "acoustic": 0.08, "optical": 0.07}  # p_drop

# Штрафи умов π_m(c): наскільки умова "гасить" відповідну модальність
PENALTY = {
    "radar": {"clear": 0.00, "rain": 0.06, "fog": 0.02, "night": 0.00, "ew_jam": 0.30},
    "rf": {"clear": 0.00, "rain": 0.02, "fog": 0.00, "night": 0.00, "ew_jam": 0.35},
    "acoustic": {"clear": 0.00, "rain": 0.20, "fog": 0.05, "night": -0.02, "ew_jam": 0.00},
    "optical": {"clear": 0.00, "rain": 0.15, "fog": 0.35, "night": 0.30, "ew_jam": 0.00},
}

CONDS = np.array(["clear", "rain", "fog", "night", "ew_jam"])
P_CONDS = [0.42, 0.18, 0.12, 0.18, 0.10]

BACKGROUND_LEVEL = 0.22  # константний шумовий рівень μ_m для фону

# --- Архітектури ухвалення рішення ---
FUSION_WEIGHTS = np.array([0.35, 0.30, 0.15, 0.20])  # фіксовані ваги w_m
LAMBDA_ADAPT = 4.0  # λ для контекстно-адаптивних ваг агентного злиття

ARCH_NAMES = ["single_radar", "static_fusion", "agentic_fusion"]
ARCH_TITLES = {
    "single_radar": "Один радарний агент",
    "static_fusion": "Статичне злиття",
    "agentic_fusion": "Агентне злиття",
}
ARCH_COLORS = {
    "single_radar": "#777777",
    "static_fusion": "#EE854A",
    "agentic_fusion": "#4878CF",
}

# --- Латентності рішення, с (логнормальні; медіана = exp(mu)) ---
LAT_MED = {"single_radar": 1.23, "static_fusion": 1.56, "agentic_fusion": 0.85}
LAT_SIG = {"single_radar": 0.40, "static_fusion": 0.35, "agentic_fusion": 0.45}

# --- Цільові рівні хибних тривог для калібрування порогів ---
FAR_TARGET = {"single_radar": 0.051, "static_fusion": 0.043, "agentic_fusion": 0.035}


@dataclass
class SimulationData:
    """Контейнер результатів однієї Monte Carlo симуляції.

    Поле ``rng`` зберігає *живий* генератор, розташований одразу після розіграшу
    латентностей, — це дозволяє наступним крокам (напр. часовому резерву в
    :func:`asf_uav.metrics.time_margin`) продовжити ту саму послідовність.
    """

    y: np.ndarray  # мітки класу (0 = фон, 1 = БпЛА)
    dist: np.ndarray  # дальність, км
    cond: np.ndarray  # умови спостереження
    scores: dict[str, np.ndarray]  # оцінки сенсорних агентів s_{i,m}
    avail: dict[str, np.ndarray]  # доступність сенсорів a_{i,m}
    S: np.ndarray  # (4, N) стек оцінок
    A: np.ndarray  # (4, N) стек доступності
    W_ag: np.ndarray  # (4, N) контекстно-адаптивні ваги
    arch: dict[str, np.ndarray]  # оцінки трьох архітектур
    latency: dict[str, np.ndarray]  # латентності трьох архітектур
    test_mask: np.ndarray  # булева маска тестової частини
    rng: np.random.Generator  # живий генератор після латентностей

    @property
    def y_test(self) -> np.ndarray:
        return self.y[self.test_mask]


def make_rng(seed: int = SEED) -> np.random.Generator:
    """Створити фіксований PCG64-генератор відтворюваності."""
    return np.random.default_rng(seed)


def _condition_penalty(modality: str, cond: np.ndarray) -> np.ndarray:
    """Вектор штрафів π_m(c_i) для заданої модальності."""
    return np.array([PENALTY[modality][c] for c in cond])


def generate_events(rng: np.random.Generator):
    """Згенерувати сирі масиви подій та оцінок сенсорів.

    Генеративна модель оцінки модальності ``m``::

        s_{i,m} = clip(mu_m(y_i, d_i, c_i) + eps_{i,m}, 0, 1),  eps ~ N(0, sigma_m^2)

    де для БпЛА ``mu_m = BASE_m - DSLOPE_m * d_i - PENALTY_m(c_i)``, а для фону
    ``mu_m = BACKGROUND_LEVEL``. Кожен сенсор із імовірністю ``P_DROP_m`` недоступний.
    """
    y = rng.integers(0, 2, N_EVENTS)  # 0 = фон, 1 = БпЛА
    dist = rng.uniform(0.5, 8.0, N_EVENTS)  # дальність, км
    cond = rng.choice(CONDS, N_EVENTS, p=P_CONDS)  # умови спостереження

    scores: dict[str, np.ndarray] = {}
    avail: dict[str, np.ndarray] = {}
    for m in MODALITIES:
        pen = _condition_penalty(m, cond)
        mu = np.where(y == 1, BASE[m] - DSLOPE[m] * dist - pen, BACKGROUND_LEVEL)
        scores[m] = np.clip(mu + rng.normal(0, NOISE[m], N_EVENTS), 0, 1)
        avail[m] = rng.random(N_EVENTS) > P_DROP[m]  # True = сенсор доступний
    return y, dist, cond, scores, avail


def adaptive_weights(cond: np.ndarray) -> np.ndarray:
    """Контекстно-адаптивні ваги агентного злиття w^{ag}_{i,m}.

    Модальність, деградована поточними умовами, експоненційно послаблюється::

        w^{ag}_{i,m} = w_m * exp(-lambda * PENALTY_m(c_i))
    """
    w_ag = np.tile(FUSION_WEIGHTS[:, None], (1, len(cond))).astype(float)
    for i, m in enumerate(MODALITIES):
        w_ag[i] *= np.exp(-LAMBDA_ADAPT * _condition_penalty(m, cond))
    return w_ag


def build_architectures(scores, avail, cond, rng: np.random.Generator):
    """Побудувати оцінки трьох архітектур та їх латентності.

    Повертає ``(S, A, W_ag, arch, latency)``. Латентності розігруються зі
    *спільного* ``rng`` у порядку :data:`ARCH_NAMES`.
    """
    S = np.stack([scores[m] for m in MODALITIES])  # (4, N)
    A = np.stack([avail[m] for m in MODALITIES])  # (4, N)
    W = FUSION_WEIGHTS[:, None]

    # Архітектура 1: один радарний агент (0, якщо радар недоступний)
    s_radar = np.where(avail["radar"], scores["radar"], 0.0)

    # Архітектура 2: статичне злиття (фіксовані ваги по доступних сенсорах)
    Wm = W * A
    s_static = (S * Wm).sum(0) / np.maximum(Wm.sum(0), 1e-9)

    # Архітектура 3: агентне злиття (контекстно-адаптивні ваги)
    W_ag = adaptive_weights(cond)
    Wm2 = W_ag * A
    s_agentic = (S * Wm2).sum(0) / np.maximum(Wm2.sum(0), 1e-9)

    arch = {
        "single_radar": s_radar,
        "static_fusion": s_static,
        "agentic_fusion": s_agentic,
    }

    latency = {
        a: rng.lognormal(np.log(LAT_MED[a]), LAT_SIG[a], N_EVENTS) for a in ARCH_NAMES
    }
    return S, A, W_ag, arch, latency


def agentic_score(S, A, W_ag, excluded: str | None = None) -> np.ndarray:
    """Оцінка агентного злиття з можливим вилученням однієї модальності (абляція)."""
    keep = [m for m in MODALITIES if m != excluded]
    idx = [MODALITIES.index(m) for m in keep]
    Wm = W_ag[idx] * A[idx]
    return (S[idx] * Wm).sum(0) / np.maximum(Wm.sum(0), 1e-9)


def simulate(seed: int = SEED) -> SimulationData:
    """Повний конвеєр генерації: події → оцінки архітектур → латентності."""
    rng = make_rng(seed)
    y, dist, cond, scores, avail = generate_events(rng)
    S, A, W_ag, arch, latency = build_architectures(scores, avail, cond, rng)
    test_mask = np.arange(N_EVENTS) >= N_EVENTS - N_TEST
    return SimulationData(
        y=y, dist=dist, cond=cond, scores=scores, avail=avail,
        S=S, A=A, W_ag=W_ag, arch=arch, latency=latency,
        test_mask=test_mask, rng=rng,
    )


def to_dataframe(sim: SimulationData) -> pd.DataFrame:
    """Зібрати повний датафрейм подій (для збереження в ``data/``)."""
    df = pd.DataFrame({"y": sim.y, "dist_km": sim.dist, "cond": sim.cond})
    for m in MODALITIES:
        df[f"s_{m}"] = sim.scores[m]
        df[f"avail_{m}"] = sim.avail[m]
    df["split"] = np.where(sim.test_mask, "test", "train")
    for a in ARCH_NAMES:
        df[f"score_{a}"] = sim.arch[a]
        df[f"lat_{a}"] = sim.latency[a]
    return df

In [ ]:
# ============================================================
# Модуль 2/3: метрики та довірчі інтервали (еквівалент src/metrics.py)
# ============================================================
"""Метрики, калібрування порогів та довірчі інтервали.

Metrics, threshold calibration and bootstrap confidence intervals.

Протокол. Поріг спрацьовування ``tau`` кожної архітектури калібрується на
**тренувальній** частині за цільовим рівнем хибних тривог (квантиль фонових
оцінок). Усі метрики обчислюються **лише на тестовій частині** (19 200 подій).
95 % CI — percentile bootstrap, :data:`N_BOOT` перевибірок тестової множини.
"""


import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, f1_score, roc_auc_score



def calibrate_threshold(
    scores: np.ndarray, y: np.ndarray, test_mask: np.ndarray, far_target: float
) -> float:
    """(1 - far_target)-квантиль фонових оцінок ТРЕНУВАЛЬНОЇ частини."""
    background_train = scores[(y == 0) & ~test_mask]
    return float(np.quantile(background_train, 1 - far_target))


def compute_metrics_table(sim: SimulationData):
    """Таблиця 4 статті: метрики + 95 % CI (bootstrap). Повертає ``(df, thresholds)``."""
    test_mask = sim.test_mask
    y = sim.y
    y_te = sim.y_test

    rows: list[dict] = []
    thresholds: dict[str, float] = {}
    boot_rng = np.random.default_rng(SEED + 1)

    for a in ARCH_NAMES:
        s = sim.arch[a]
        tau = calibrate_threshold(s, y, test_mask, FAR_TARGET[a])
        thresholds[a] = tau

        s_te = s[test_mask]
        pred = s_te >= tau
        lat_te = sim.latency[a][test_mask]

        pd_ = pred[y_te == 1].mean()
        far = pred[y_te == 0].mean()
        prec = pred[(y_te == 1) & pred].size / max(pred.sum(), 1)
        f1 = f1_score(y_te, pred)
        auc = roc_auc_score(y_te, s_te)
        brier = brier_score_loss(y_te, np.clip(s_te, 0, 1))

        # --- percentile bootstrap для Pd і FAR ---
        n = len(y_te)
        pd_b, far_b = [], []
        for _ in range(N_BOOT):
            idx = boot_rng.integers(0, n, n)
            yb, pb = y_te[idx], pred[idx]
            pd_b.append(pb[yb == 1].mean())
            far_b.append(pb[yb == 0].mean())
        lo_pd, hi_pd = np.percentile(pd_b, [2.5, 97.5])
        lo_far, hi_far = np.percentile(far_b, [2.5, 97.5])

        rows.append(
            dict(
                architecture=a, threshold=tau, precision=prec, pd=pd_, f1=f1, far=far,
                roc_auc=auc, brier=brier,
                pd_ci_lo=lo_pd, pd_ci_hi=hi_pd, far_ci_lo=lo_far, far_ci_hi=hi_far,
                latency_median_s=np.median(lat_te),
                latency_p95_s=np.percentile(lat_te, 95),
            )
        )

    return pd.DataFrame(rows), thresholds


def pd_by_distance(sim: SimulationData, thresholds: dict[str, float]) -> pd.DataFrame:
    """Pd БпЛА за кілометровими інтервалами дальності (рис. 3)."""
    bins = np.arange(0.5, 8.5 + 1e-9, 1.0)
    labels = [f"{a:.1f}–{b:.1f}" for a, b in zip(bins[:-1], bins[1:])]
    d_te = sim.dist[sim.test_mask]
    y_te = sim.y_test

    rows: list[dict] = []
    for a in ARCH_NAMES:
        pred = sim.arch[a][sim.test_mask] >= thresholds[a]
        for j in range(len(bins) - 1):
            sel = (y_te == 1) & (d_te >= bins[j]) & (d_te < bins[j + 1])
            rows.append(
                dict(
                    architecture=a, dist_bin=labels[j],
                    bin_center=(bins[j] + bins[j + 1]) / 2,
                    n=int(sel.sum()), pd=pred[sel].mean(),
                )
            )
    return pd.DataFrame(rows)


def metrics_by_condition(
    sim: SimulationData, thresholds: dict[str, float]
) -> pd.DataFrame:
    """Pd та FAR кожної архітектури окремо для кожної умови спостереження (рис. 5а)."""
    c_te = sim.cond[sim.test_mask]
    y_te = sim.y_test

    rows: list[dict] = []
    for a in ARCH_NAMES:
        pred = sim.arch[a][sim.test_mask] >= thresholds[a]
        for c in CONDS:
            sel = (y_te == 1) & (c_te == c)
            self_far = pred[(y_te == 0) & (c_te == c)].mean()
            rows.append(dict(architecture=a, cond=c, pd=pred[sel].mean(), far=self_far))
    return pd.DataFrame(rows)


def ablation(sim: SimulationData) -> pd.DataFrame:
    """Абляція агентного злиття: по черзі вилучаємо одну модальність (рис. 5)."""

    y = sim.y
    y_te = sim.y_test
    test_mask = sim.test_mask

    rows: list[dict] = []
    for ex in [None, *MODALITIES]:
        s = agentic_score(sim.S, sim.A, sim.W_ag, excluded=ex)
        tau = calibrate_threshold(s, y, test_mask, FAR_TARGET["agentic_fusion"])
        pred = s[test_mask] >= tau
        rows.append(
            dict(
                config="повна система" if ex is None else f"без {ex}",
                f1=f1_score(y_te, pred),
                pd=pred[y_te == 1].mean(),
                far=pred[y_te == 0].mean(),
            )
        )
    return pd.DataFrame(rows)


def time_margin(sim: SimulationData, thresholds: dict[str, float]) -> pd.DataFrame:
    """Часовий резерв до цілі після машинного рішення та людського підтвердження.

    Використовує *живий* ``sim.rng`` (розташований після розіграшу латентностей),
    щоб зберегти точну послідовність відтворюваності оригінального ноутбука.
    """
    rng = sim.rng
    test_mask = sim.test_mask
    y_te = sim.y_test
    d_te = sim.dist[test_mask]

    pred_ag = sim.arch["agentic_fusion"][test_mask] >= thresholds["agentic_fusion"]
    det = (y_te == 1) & pred_ag  # виявлені БпЛА в тесті

    v = rng.uniform(15, 30, det.sum())  # швидкість цілі, м/с
    t_arr = 1000 * d_te[det] / v  # час підльоту, с
    lat_det = sim.latency["agentic_fusion"][test_mask][det]  # латентність машини
    t_human = rng.lognormal(np.log(6.0), 0.5, det.sum())  # час підтвердження оператором

    return pd.DataFrame(
        {
            "dist_km": d_te[det],
            "speed_ms": v,
            "t_arrival_s": t_arr,
            "margin_after_machine_s": t_arr - lat_det,
            "margin_after_human_s": t_arr - lat_det - t_human,
        }
    )

In [ ]:
# ============================================================
# Модуль 3/3: побудова рисунків (еквівалент src/make_figures.py)
# ============================================================
"""Побудова всіх рисунків статті засобами matplotlib (inline).

Figure generation for the reproducibility package.

Кожна функція приймає вже обчислені дані (об'єкт симуляції або датафрейм метрик)
і повертає :class:`matplotlib.figure.Figure`. За переданого ``save`` рисунок також
зберігається у файл (320 dpi). Модуль не викликає ``plt.show()`` — це залишено
ноутбуку, щоб функції були придатні для тестів у безголовому режимі.
"""


import matplotlib.pyplot as plt
import numpy as np



def setup_matplotlib() -> None:
    """Уніфіковані параметри оформлення рисунків."""
    plt.rcParams.update(
        {
            "figure.dpi": 110,
            "savefig.dpi": 320,
            "font.size": 10,
            "axes.grid": True,
            "grid.alpha": 0.3,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


def _save(fig, save: str | None) -> None:
    if save:
        fig.savefig(save, bbox_inches="tight")


# ------------------------------------------------------------------
# Рис. 1. Візуальний огляд датасету
# ------------------------------------------------------------------
def fig_dataset_overview(sim: SimulationData, save: str | None = None):
    df_dist, df_cond = sim.dist, sim.cond
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))

    axes[0, 0].hist(df_dist, bins=40, color="#4878CF", alpha=0.85)
    axes[0, 0].set_title("(a) Дальність подій")
    axes[0, 0].set_xlabel("км")

    vc = np.array([(df_cond == c).sum() for c in CONDS])
    axes[0, 1].bar(CONDS, vc, color="#6ACC65")
    axes[0, 1].set_title("(b) Умови спостереження")
    axes[0, 1].tick_params(axis="x", rotation=30)

    av = [sim.avail[m].mean() * 100 for m in MODALITIES]
    axes[0, 2].bar(MODALITIES, av, color="#D65F5F")
    axes[0, 2].set_ylim(85, 100)
    axes[0, 2].set_title("(c) Доступність сенсорів, %")

    for ax, m in zip(axes[1], MODALITIES[:3]):
        ax.hist(sim.scores[m][sim.y == 0], bins=50, alpha=0.6, label="фон",
                color="#777777", density=True)
        ax.hist(sim.scores[m][sim.y == 1], bins=50, alpha=0.6, label="БпЛА",
                color="#EE854A", density=True)
        ax.set_title(f"Оцінка агента: {m}")
        ax.legend()
    fig.tight_layout()
    _save(fig, save)
    return fig


def fig_optical_scores(sim: SimulationData, save: str | None = None):
    """Розподіл оцінок оптики (не вміщається у сітку 2×3 рис. 1)."""
    fig = plt.figure(figsize=(4.4, 2.9))
    plt.hist(sim.scores["optical"][sim.y == 0], bins=50, alpha=0.6, label="фон",
             color="#777777", density=True)
    plt.hist(sim.scores["optical"][sim.y == 1], bins=50, alpha=0.6, label="БпЛА",
             color="#EE854A", density=True)
    plt.title("Оцінка агента: optical")
    plt.legend()
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 2. Розділюваність класів за архітектурами
# ------------------------------------------------------------------
def fig_score_separability(sim: SimulationData, save: str | None = None):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, a in zip(axes, ARCH_NAMES):
        s = sim.arch[a]
        ax.hist(s[sim.y == 0], bins=60, alpha=0.6, density=True, label="фон", color="#777777")
        ax.hist(s[sim.y == 1], bins=60, alpha=0.6, density=True, label="БпЛА", color="#4878CF")
        ax.set_title(ARCH_TITLES[a])
        ax.set_xlabel("оцінка впевненості")
        ax.legend()
    axes[0].set_ylabel("щільність")
    fig.suptitle("Що менше перекриття розподілів — то краща розділюваність", y=1.04)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. до Таблиці 4. Pd і FAR з довірчими інтервалами
# ------------------------------------------------------------------
def fig_metrics_bars(metrics, save: str | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    xs = np.arange(len(metrics))
    labels = ["Радар", "Статичне", "Агентне"]
    specs = [
        ("pd", "pd_ci_lo", "pd_ci_hi", "Ймовірність виявлення Pd", "#4878CF"),
        ("far", "far_ci_lo", "far_ci_hi", "Рівень хибних тривог FAR", "#D65F5F"),
    ]
    for ax, (col, lo, hi, ttl, clr) in zip(axes, specs):
        err = np.vstack([metrics[col] - metrics[lo], metrics[hi] - metrics[col]])
        ax.bar(xs, metrics[col], yerr=err, capsize=5, color=clr, alpha=0.85)
        ax.set_xticks(xs)
        ax.set_xticklabels(labels)
        ax.set_title(ttl + " (95 % CI)")
        for x, v in zip(xs, metrics[col]):
            ax.text(x, v, f" {v:.3f}", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 3. Pd за дальністю
# ------------------------------------------------------------------
def fig_pd_by_distance(dist_tab, save: str | None = None):
    fig = plt.figure(figsize=(7.5, 4))
    for a in ARCH_NAMES:
        t = dist_tab[dist_tab.architecture == a]
        plt.plot(t.bin_center, t.pd, "o-", color=ARCH_COLORS[a], label=ARCH_TITLES[a])
    plt.xlabel("Дальність, км")
    plt.ylabel("Pd")
    plt.ylim(0, 1.02)
    plt.title("Рис. 3. Ймовірність виявлення за дальністю (тестова частина)")
    plt.legend()
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 4. ROC-криві та робочі точки
# ------------------------------------------------------------------
def fig_roc(sim: SimulationData, metrics, save: str | None = None):
    from sklearn.metrics import roc_auc_score, roc_curve

    y_te = sim.y_test
    fig = plt.figure(figsize=(6.2, 5))
    for a in ARCH_NAMES:
        s_te = sim.arch[a][sim.test_mask]
        fpr, tpr, _ = roc_curve(y_te, s_te)
        auc = roc_auc_score(y_te, s_te)
        plt.plot(fpr, tpr, color=ARCH_COLORS[a], label=f"{ARCH_TITLES[a]} (AUC={auc:.3f})")
        row = metrics[metrics.architecture == a].iloc[0]
        plt.plot(row.far, row.pd, "o", ms=9, mec="k", color=ARCH_COLORS[a])
    plt.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
    plt.xlim(0, 0.25)
    plt.ylim(0.4, 1.005)
    plt.xlabel("FAR")
    plt.ylabel("Pd")
    plt.title("Рис. 4. Компроміс FAR–Pd; маркери — робочі точки з Таблиці 4")
    plt.legend(loc="lower right")
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 5а. Теплокарта Pd за умовами
# ------------------------------------------------------------------
def fig_pd_by_condition(cond_tab, save: str | None = None):
    piv = cond_tab.pivot(index="cond", columns="architecture", values="pd").reindex(CONDS)
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    im = ax.imshow(piv[ARCH_NAMES].values, cmap="RdYlGn", vmin=0.3, vmax=1.0, aspect="auto")
    ax.set_xticks(range(3))
    ax.set_xticklabels(["Радар", "Статичне", "Агентне"])
    ax.set_yticks(range(len(CONDS)))
    ax.set_yticklabels(CONDS)
    for i in range(len(CONDS)):
        for j, col in enumerate(ARCH_NAMES):
            ax.text(j, i, f"{piv[col].iloc[i]:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="Pd")
    ax.set_title("Pd за умовами спостереження")
    ax.grid(False)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 5. Абляційний аналіз
# ------------------------------------------------------------------
def fig_ablation(abl, save: str | None = None):
    fig = plt.figure(figsize=(7, 3.6))
    colors = ["#4878CF"] + ["#D65F5F"] * 4
    bars = plt.barh(abl.config[::-1], abl.f1[::-1], color=colors[::-1], alpha=0.9)
    plt.xlabel("F1 (цільовий FAR = 3,5 %)")
    plt.xlim(0.8, 1.0)
    plt.title("Рис. 5. Абляційний аналіз агентного злиття")
    for b, v in zip(bars, abl.f1[::-1]):
        plt.text(v, b.get_y() + b.get_height() / 2, f" {v:.3f}", va="center", fontsize=9)
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 6. Boxplot латентностей
# ------------------------------------------------------------------
def fig_latency_boxplot(sim: SimulationData, save: str | None = None):
    lat_data = [sim.latency[a][sim.test_mask] for a in ARCH_NAMES]
    fig = plt.figure(figsize=(7, 4))
    bp = plt.boxplot(lat_data, tick_labels=["Радар", "Статичне", "Агентне"],
                     showfliers=False, patch_artist=True, widths=0.55)
    for patch, a in zip(bp["boxes"], ARCH_NAMES):
        patch.set_facecolor(ARCH_COLORS[a])
        patch.set_alpha(0.7)
    for i, a in enumerate(ARCH_NAMES):
        med = np.median(sim.latency[a][sim.test_mask])
        plt.text(i + 1, med, f"  медіана {med:.2f} с", va="center", fontsize=9)
    plt.ylabel("Латентність рішення, с")
    plt.title("Рис. 6. Розподіл латентності за архітектурами (без викидів)")
    fig.tight_layout()
    _save(fig, save)
    return fig


# ------------------------------------------------------------------
# Рис. 7. Часовий резерв
# ------------------------------------------------------------------
def fig_time_margin(tm, save: str | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].hist(tm.margin_after_machine_s, bins=60, alpha=0.65, label="після машини", color="#4878CF")
    axes[0].hist(tm.margin_after_human_s, bins=60, alpha=0.65, label="після людини", color="#EE854A")
    axes[0].set_xlabel("Часовий резерв, с")
    axes[0].set_ylabel("К-сть подій")
    axes[0].set_title("Гістограма часового резерву")
    axes[0].legend()

    for col, lbl, c in [
        ("margin_after_machine_s", "після машини", "#4878CF"),
        ("margin_after_human_s", "після людини", "#EE854A"),
    ]:
        xs = np.sort(tm[col])
        axes[1].plot(xs, np.linspace(0, 1, len(xs)), color=c, label=lbl)
    axes[1].set_xlabel("Часовий резерв, с")
    axes[1].set_ylabel("F(x)")
    axes[1].set_title("Емпірична функція розподілу (CDF)")
    axes[1].legend()
    fig.tight_layout()
    _save(fig, save)
    return fig

## 0.1. Налаштування середовища / Environment setup

Фіксуємо генератор випадкових чисел (`SEED = 20260`), уніфікуємо оформлення рисунків
і створюємо робочі каталоги `data/`, `results/`, `figures/`.

In [ ]:
import os, json, hashlib, warnings
from IPython.display import display

warnings.filterwarnings("ignore")
setup_matplotlib()

for d in ("data", "results", "figures"):
    os.makedirs(d, exist_ok=True)

print(f"Seed: {SEED} | events: {N_EVENTS} | test: {N_TEST} | modalities: {MODALITIES}")
print("NumPy:", np.__version__, "| pandas:", pd.__version__)

## 1. Генерація синтетичного датасету / Synthetic dataset generation

Кожна подія $i$ описується:

* міткою класу $y_i \in \{0, 1\}$ — «фон» (птах, завада, цивільний ЛА) або «БпЛА»;
* дальністю $d_i \sim U(0{,}5;\; 8)$ км;
* умовами спостереження $c_i \in \{\text{clear, rain, fog, night, ew\_jam}\}$;
* оцінками впевненості чотирьох сенсорних агентів $s_{i,m} \in [0,1]$.

**Генеративна модель оцінки** модальності $m$:

$$s_{i,m} = \mathrm{clip}\Big( \mu_m(y_i, d_i, c_i) + \varepsilon_{i,m},\; 0,\; 1 \Big), \qquad \varepsilon_{i,m} \sim \mathcal{N}(0, \sigma_m^2)$$

де для БпЛА $\mu_m = \beta_m - \alpha_m d_i - \pi_m(c_i)$, а для фону $\mu_m = 0{,}22$.
Крім того, кожен сенсор із імовірністю $p^{drop}_m$ **недоступний** (відмова каналу).

| Модальність | Найчутливіша до | $p^{drop}_m$ |
|---|---|---|
| Радар | РЕБ-придушення (`ew_jam`) | 4 % |
| РЧ-аналізатор | РЕБ-придушення | 6 % |
| Акустика | Дощ (шум крапель) | 8 % |
| Оптика | Туман, ніч | 7 % |

In [ ]:
sim = simulate(SEED)          # повний конвеєр: події → оцінки → латентності
df  = to_dataframe(sim)       # повний датафрейм подій
df.to_csv("data/synthetic_events.csv", index=False)

y, dist, cond = sim.y, sim.dist, sim.cond
print(f"Згенеровано {len(df):,} подій | БпЛА: {y.mean():.1%} | тест: {(df.split=='test').sum():,}")
df.head()

### 1.1. Огляд датасету / Dataset overview

Перевіряємо, що датасет виглядає «здоровим»: рівномірна дальність, задані частки умов,
і що розподіли оцінок сенсорів для БпЛА і фону перекриваються, але розділювані.

In [ ]:
fig_dataset_overview(sim, save="figures/fig1_dataset_overview.png"); plt.show()
fig_optical_scores(sim); plt.show()

## 2. Три архітектури ухвалення рішення / Three decision architectures

1. **Один радарний агент (baseline).** Рішення лише за $s_{radar}$; якщо радар недоступний — оцінка 0.
2. **Статичне злиття.** Зважене середнє з **фіксованими** вагами $w = (0{,}35;\,0{,}30;\,0{,}15;\,0{,}20)$ по доступних сенсорах:
$$s^{stat}_i = \frac{\sum_m w_m a_{i,m} s_{i,m}}{\sum_m w_m a_{i,m}}$$
3. **Агентне злиття (запропонована архітектура).** Агент-координатор **адаптує ваги до контексту**:
$$w^{ag}_{i,m} = w_m \exp\big(-\lambda\, \pi_m(c_i)\big), \qquad \lambda = 4$$

Оцінки й латентності всіх трьох архітектур обчислено всередині `simulate()`. Латентність
моделюється логнормально: агентний конвеєр — медіана ≈ 0,85 с, статичне злиття ≈ 1,56 с,
одиночний радар ≈ 1,23 с.

In [ ]:
for a in ARCH_NAMES:
    s = sim.arch[a]
    print(f"{ARCH_TITLES[a]:20s}: score∈[{s.min():.3f},{s.max():.3f}] "
          f"| медіана латентності ≈ {np.median(sim.latency[a]):.2f} с")

In [ ]:
fig_score_separability(sim, save="figures/fig2_score_separability.png"); plt.show()

## 3. Метрики та довірчі інтервали / Metrics, thresholds, bootstrap CI

Поріг $\tau$ кожної архітектури калібрується **на тренувальній частині** за цільовим FAR
(5,1 % / 4,3 % / 3,5 %). Метрики обчислюються **лише на тесті** (19 200 подій):
Pd, FAR, Precision, F1, ROC AUC, Brier, медіана та P95 латентності. 95 % CI —
percentile bootstrap, 800 перевибірок.

In [ ]:
metrics, thresholds = compute_metrics_table(sim)
metrics.to_csv("results/table4_metrics.csv", index=False)
metrics[["architecture","precision","pd","f1","far","roc_auc","brier",
         "latency_median_s","latency_p95_s"]].round(4)

### Таблиця 4 статті (демонстраційне відтворення)

Очікувані значення: агентне злиття Pd ≈ 93 %, F1 ≈ 95 %, FAR ≈ 3,5 %; статичне злиття
Pd ≈ 92 %, FAR ≈ 4,2 %; один радар Pd ≈ 65 %, FAR ≈ 5,1 %. Нижче — Pd/FAR із 95 % CI.

In [ ]:
ci_cols = [c for c in metrics.columns if c.endswith(("_ci_lo", "_ci_hi"))]
display(metrics.set_index("architecture")[["pd", "far"] + ci_cols].round(4).T)
fig_metrics_bars(metrics, save="figures/fig_table4_bars.png"); plt.show()

## 4. Стійкість за дистанцією (рис. 3) / Pd vs distance

Розбиваємо тестові події-БпЛА на кілометрові інтервали й оцінюємо Pd. Перевага агентного
злиття зростає з дальністю, бо на великих дистанціях окремі канали деградують.

In [ ]:
dist_tab = pd_by_distance(sim, thresholds)
dist_tab.to_csv("results/pd_by_distance.csv", index=False)
fig_pd_by_distance(dist_tab, save="figures/fig3_pd_by_distance.png"); plt.show()
dist_tab.pivot(index="dist_bin", columns="architecture", values="pd").round(4)

## 5. Компроміс FAR–Pd (рис. 4) / ROC and operating points

Повні ROC-криві показують, що агентне злиття домінує на всьому діапазоні порогів,
а не лише в обраній робочій точці (маркери).

In [ ]:
fig_roc(sim, metrics, save="figures/fig4_far_pd_tradeoff.png"); plt.show()

## 6. Метрики за умовами спостереження та абляційний аналіз (рис. 5)

**За умовами:** Pd кожної архітектури для clear / rain / fog / night / ew_jam.
**Абляція:** по черзі вилучаємо одну модальність, повторно калібруємо поріг (FAR = 3,5 %)
і вимірюємо F1 — це показує внесок кожного сенсора.

In [ ]:
cond_tab = metrics_by_condition(sim, thresholds)
cond_tab.to_csv("results/metrics_by_condition.csv", index=False)
piv = cond_tab.pivot(index="cond", columns="architecture", values="pd").reindex(CONDS)
display(piv.round(4))
fig_pd_by_condition(cond_tab, save="figures/fig5a_pd_by_condition.png"); plt.show()

In [ ]:
abl = ablation(sim)
abl.to_csv("results/ablations.csv", index=False)
display(abl.round(4))
fig_ablation(abl, save="figures/fig5_ablation_f1.png"); plt.show()

## 7. Латентність рішення (рис. 6) / Decision latency

Boxplot латентностей на тесті. Агентний конвеєр ухвалює рішення раніше завдяки
адаптивному гейтуванню каналів (медіана ≈ 0,85 с проти 1,56 с у статичного злиття).

In [ ]:
fig_latency_boxplot(sim, save="figures/fig6_latency_boxplot.png"); plt.show()

## 8. Часовий резерв після машинного рішення та людського підтвердження

Для кожної **виявленої** цілі (агентне злиття) оцінюємо часовий резерв до об'єкта:

$$T^{arrival}_i = \frac{1000\, d_i}{v_i}, \quad v_i \sim U(15; 30)\ \text{м/с}$$

* після машини: $M^{mach}_i = T^{arrival}_i - L_i$;
* після людини: $M^{hum}_i = M^{mach}_i - T^{conf}_i$, де $T^{conf}$ — логнормальний
  час підтвердження оператором (медіана ≈ 6 с). Це реалізує протокол «людина в контурі».

In [ ]:
tm = time_margin(sim, thresholds)
tm.to_csv("results/time_margin.csv", index=False)

print("Медіана резерву після машинного рішення, с:", round(float(np.median(tm.margin_after_machine_s)), 2))
print("Медіана після людського підтвердження, с:  ", round(float(np.median(tm.margin_after_human_s)), 2))
print("P10 / P90 (після людини), с:",
      round(float(np.percentile(tm.margin_after_human_s, 10)), 2), "/",
      round(float(np.percentile(tm.margin_after_human_s, 90)), 2))

fig_time_margin(tm, save="figures/fig7_time_margin.png"); plt.show()

## 9. Збережені артефакти та контрольні суми / Saved artifacts & checksums

Усі таблиці — в `data/` та `results/`, рисунки (320 dpi) — у `figures/`. SHA-256
контрольні суми фіксують точний вміст файлів. У Colab їх можна завантажити через панель
*Files* ліворуч або `files.download(...)`.

In [ ]:
manifest = {}
for root in ("data", "results", "figures"):
    for f in sorted(os.listdir(root)):
        p = os.path.join(root, f)
        manifest[p] = hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
with open("results/MANIFEST_sha256.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
for p, h in manifest.items():
    print(f"{h}  {p}  ({os.path.getsize(p)/1024:.1f} КБ)")

## 10. Обмеження / Limitations

1. **Синтетичні дані:** розподіли оцінок, частки відмов і фонові класи задані авторами, а не з польових вимірювань.
2. **Подієва незалежність:** не моделюються часові кореляції, рої БпЛА, маскування, рельєф, багатопроменевість.
3. **Спрощений людський компонент:** час підтвердження — логнормальний розподіл без урахування втоми/стресу/ергономіки.
4. **Немає прямого емпіричного порівняння** з опублікованими системами виявлення.
5. Наведені числа **не можна використовувати** для закупівель чи нормування часу евакуації.

Ліцензія: MIT. Модульна версія з `pytest`-тестами — у каталозі `asf-uav-warning/` репозиторію.